<a href="https://colab.research.google.com/github/ZaAth0/rag-sales-mistral/blob/main/AgenteRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
#Solo para github
import json
import nbformat

In [52]:
# Instalar librerías necesarias
!pip install -q mistralai langchain langchain-mistralai chromadb sentence-transformers pandas langchain_community langchain-text-splitters langchain-huggingface

# Reiniciar runtime si es necesario (opcional, pero recomendado)
#import os
#os.kill(os.getpid(), 9)
#!pip install mistralai

Configurar ApiKey

In [53]:
# Importar librerías necesarias
from google.colab import userdata
import os

# Obtener API key desde los secrets de Colab
try:
    # Tu secret se llama "PruebaIA" según configuraste
    API_KEY = userdata.get('PruebaIA')
    print(f"✓ API Key recuperada de los secrets de Colab")
    print(f"  Key: {API_KEY[:10]}...{API_KEY[-5:]}")
except Exception as e:
    print(f"❌ Error: No se encontró el secret 'PruebaIA'")
    print("  Por favor, configura el secret en Colab: 🔑 > Add secret")
    print("  Nombre: PruebaIA")
    print("  Valor: briQ1ImIMb7yQChYsJEVqU5WDaNh9SrG")
    raise

# Configurar como variable de entorno (opcional, buena práctica)
os.environ["MISTRAL_API_KEY"] = API_KEY

print("✓ API Key configurada correctamente")

✓ API Key recuperada de los secrets de Colab
  Key: briQ1ImIMb...h9SrG
✓ API Key configurada correctamente


Configurar conexion a Mistral

In [54]:
from mistralai.client import Mistral

# Inicializar cliente con la API key desde secrets
client = Mistral(api_key=API_KEY)

# Probar conexión básica
print("🔌 Probando conexión con Mistral API...")
try:
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {"role": "user", "content": "Responde solo 'Conexión exitosa con Mistral' si puedes leer este mensaje"}
        ],
        temperature=0.7,
        max_tokens=50
    )
    print("✓ Conexión exitosa con Mistral API")
    print(f"📨 Respuesta de prueba: {response.choices[0].message.content}")

    # Verificar modelo disponible
    print(f"✓ Modelo utilizado: mistral-small-latest")

except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("  Verifica que la API key sea válida y tenga créditos disponibles")

🔌 Probando conexión con Mistral API...
✓ Conexión exitosa con Mistral API
📨 Respuesta de prueba: Conexión exitosa con Mistral
✓ Modelo utilizado: mistral-small-latest


Preparar el Dataset para el RAG

In [55]:
import pandas as pd
import chardet
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Cargar tu dataset (asumiendo que está en el entorno)
print("📂 Cargando dataset...")

# Detectar codificación automáticamente
with open('sales_data_normalizado.csv', 'rb') as f:
    result = chardet.detect(f.read())
    encoding = result['encoding']
    print(f"  Codificación detectada: {encoding}")

# Cargar dataset
df = pd.read_csv('sales_data_normalizado.csv', encoding=encoding)

print(f"✓ Dataset cargado exitosamente")
print(f"  Dimensiones: {df.shape}")
print(f"  Columnas: {list(df.columns[:10])}...")

# Mostrar primeras filas para verificar
print(f"\n📊 Muestra de datos:")
print(df.head(3)[['ORDERNUMBER', 'SALES', 'COUNTRY', 'STATUS']])

# Crear documentos a partir del CSV para RAG
def crear_documentos_ventas(df, max_registros=1000):
    """
    Convierte cada registro del dataset en un documento de texto enriquecido
    para el sistema RAG.
    """
    documentos = []

    for idx, row in df.iterrows():
        # Crear un texto natural estructurado por cada venta
        doc_text = f"""
=== VENTA #{row['ORDERNUMBER']} ===
Producto: {row.get('PRODUCTLINE', 'No especificado')}
Línea de producto: {row.get('PRODUCTLINE', 'N/A')}
Cantidad: {row['QUANTITYORDERED']} unidades
Precio unitario: ${row['PRICEEACH']:.2f}
Venta total: ${row['SALES']:.2f}
Fecha: {row['ORDERDATE']}
Estado del pedido: {row['STATUS']}
Trimestre: {row['QTR_ID']}
Mes: {row['MONTH_ID']}
Año: {row['YEAR_ID']}
Cliente: {row.get('CUSTOMERNAME', 'No especificado')}
País: {row.get('COUNTRY', 'No especificado')}
Ciudad: {row.get('CITY', 'No especificado')}
Territorio: {row.get('TERRITORY', 'No especificado')}
Tamaño del deal: {row.get('DEALSIZE', 'No especificado')}
"""

        # Crear metadatos útiles para filtrar después
        metadata = {
            "order_number": int(row['ORDERNUMBER']),
            "sales": float(row['SALES']),
            "country": str(row.get('COUNTRY', 'Unknown')),
            "status": str(row['STATUS']),
            "product_line": str(row.get('PRODUCTLINE', 'Unknown')),
            "deal_size": str(row.get('DEALSIZE', 'Unknown')),
            "year": int(row['YEAR_ID']) if pd.notna(row['YEAR_ID']) else 0
        }

        documentos.append({
            "text": doc_text.strip(),
            "metadata": metadata
        })

        # Limitar para demo (puedes aumentar o quitar)
        if idx >= max_registros - 1:
            break

    print(f"✓ Documentos creados: {len(documentos)} registros procesados")
    return documentos

# Crear documentos
documentos = crear_documentos_ventas(df, max_registros=500)

📂 Cargando dataset...
  Codificación detectada: ascii
✓ Dataset cargado exitosamente
  Dimensiones: (147, 25)
  Columnas: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID']...

📊 Muestra de datos:
   ORDERNUMBER     SALES    COUNTRY   STATUS
0     0.343333  0.363235  Australia  Shipped
1     0.803333  0.087555  Australia  Shipped
2     0.500000  0.466256  Australia  Shipped
✓ Documentos creados: 147 registros procesados


Configurar el Vector Store para el RAG

In [56]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

print("🔧 Configurando Vector Store...")

# Usar embeddings gratuitos de código abierto (no requieren API key)
# Estos convierten texto en vectores numéricos para búsqueda semántica
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},  # Usamos CPU para no consumir GPU
    encode_kwargs={'normalize_embeddings': True}
)

print("✓ Modelo de embeddings cargado")

# Convertir a formato LangChain
langchain_docs = [
    Document(page_content=doc["text"], metadata=doc["metadata"])
    for doc in documentos
]

# Crear base de datos vectorial
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    persist_directory="./chroma_sales_db"
)

print(f"✓ Vector store creado exitosamente")
print(f"  Documentos indexados: {vectorstore._collection.count()}")
print(f"  Dimensión de embeddings: {len(embeddings.embed_query('test'))}")

🔧 Configurando Vector Store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo de embeddings cargado
✓ Vector store creado exitosamente
  Documentos indexados: 588
  Dimensión de embeddings: 384


Configurar Pipeline RAG completo con Mistral

In [57]:
from mistralai.client import Mistral
from typing import List, Dict
import json

class RAGSystemVentas:
    """
    Sistema RAG especializado en análisis de ventas usando Mistral API
    """

    def __init__(self, vectorstore, api_key, model="mistral-small-latest"):
        self.vectorstore = vectorstore
        self.client = Mistral(api_key=api_key)
        self.model = model
        self.historial_consultas = []

    def buscar_documentos_relevantes(self, pregunta: str, k: int = 3) -> tuple:
        """
        Busca los documentos más relevantes para la pregunta
        """
        # Búsqueda de similitud semántica
        docs = self.vectorstore.similarity_search(pregunta, k=k)

        # Extraer contexto y fuentes
        contexto = "\n\n---\n\n".join([doc.page_content for doc in docs])
        fuentes = [{
            "order_number": doc.metadata.get('order_number'),
            "country": doc.metadata.get('country'),
            "sales": doc.metadata.get('sales'),
            "status": doc.metadata.get('status')
        } for doc in docs]

        return contexto, fuentes

    def generar_respuesta(self, pregunta: str, contexto: str) -> str:
        """
        Genera respuesta usando Mistral API con el contexto recuperado
        """
        prompt_rag = f"""Eres un analista de datos de ventas experto. Debes responder preguntas basándote ESTRICTAMENTE en el contexto proporcionado.

CONTEXTO DE VENTAS:
{contexto}

PREGUNTA DEL USUARIO:
{pregunta}

INSTRUCCIONES:
1. Usa SOLO la información del contexto para responder
2. Si el contexto no contiene suficiente información, responde: "No tengo suficiente información en los datos para responder esto"
3. Cita números, montos y fechas específicos cuando estén disponibles
4. Sé conciso pero completo en tu respuesta
5. Si hay múltiples ejemplos, menciona los más relevantes

RESPUESTA:"""

        try:
            response = self.client.chat.complete(
                model=self.model,
                messages=[
                    {"role": "system", "content": "Eres un asistente experto en análisis de datos de ventas. Siempre basas tus respuestas en datos concretos del contexto proporcionado."},
                    {"role": "user", "content": prompt_rag}
                ],
                temperature=0.3,  # Bajo para respuestas factuales
                max_tokens=600
            )

            respuesta = response.choices[0].message.content
            return respuesta

        except Exception as e:
            return f"Error al generar respuesta: {e}"

    def consultar(self, pregunta: str, mostrar_contexto: bool = False) -> Dict:
        """
        Realiza una consulta completa al sistema RAG
        """
        # Paso 1: Recuperar contexto relevante
        contexto, fuentes = self.buscar_documentos_relevantes(pregunta)

        # Paso 2: Generar respuesta con Mistral
        respuesta = self.generar_respuesta(pregunta, contexto)

        # Paso 3: Guardar en historial
        consulta = {
            "pregunta": pregunta,
            "respuesta": respuesta,
            "fuentes": fuentes,
            "timestamp": pd.Timestamp.now().isoformat()
        }
        self.historial_consultas.append(consulta)

        # Paso 4: Preparar resultado
        resultado = {
            "pregunta": pregunta,
            "respuesta": respuesta,
            "fuentes": fuentes,
            "contexto_usado": contexto if mostrar_contexto else None
        }

        return resultado

    def obtener_estadisticas(self) -> Dict:
        """
        Retorna estadísticas del sistema RAG
        """
        return {
            "total_consultas": len(self.historial_consultas),
            "modelo": self.model,
            "documentos_indexados": self.vectorstore._collection.count()
        }

# Inicializar sistema RAG
print("🚀 Inicializando sistema RAG...")
rag_system = RAGSystemVentas(
    vectorstore=vectorstore,
    api_key=API_KEY,
    model="mistral-small-latest"
)

print("✓ Sistema RAG inicializado exitosamente")
print(f"  Modelo: mistral-small-latest")
print(f"  Documentos indexados: {rag_system.obtener_estadisticas()['documentos_indexados']}")

🚀 Inicializando sistema RAG...
✓ Sistema RAG inicializado exitosamente
  Modelo: mistral-small-latest
  Documentos indexados: 588


Probar el sistema con Ejemplos Reales

In [58]:
print("🧪 PROBANDO SISTEMA RAG CON PREGUNTAS REALES")
print("="*60)

# Lista de preguntas de prueba sobre ventas
preguntas_prueba = [
    "¿Cuál fue la venta de mayor valor y en qué país se realizó?",
    "¿Qué países tienen ventas registradas y cuál es el monto promedio por país?",
    "¿Cuál es el estado más común de los pedidos?",
    "Muestra las ventas del tipo 'Small' deal size",
    "¿Qué productos se han vendido y en qué cantidades?"
]

# Probar cada pregunta
for i, pregunta in enumerate(preguntas_prueba, 1):
    print(f"\n{'='*60}")
    print(f"📝 PRUEBA {i}: {pregunta}")
    print(f"{'='*60}")

    resultado = rag_system.consultar(pregunta, mostrar_contexto=False)

    print(f"\n🤖 RESPUESTA:\n{resultado['respuesta']}")
    print(f"\n📚 FUENTES UTILIZADAS: {len(resultado['fuentes'])} documentos")

    # Mostrar primeras fuentes como ejemplo
    for j, fuente in enumerate(resultado['fuentes'][:2], 1):
        print(f"   Fuente {j}: Orden #{fuente.get('order_number', 'N/A')} - {fuente.get('country', 'N/A')} - ${fuente.get('sales', 0):,.2f}")

print("\n" + "="*60)
print("✅ PRUEBAS COMPLETADAS EXITOSAMENTE")

🧪 PROBANDO SISTEMA RAG CON PREGUNTAS REALES

📝 PRUEBA 1: ¿Cuál fue la venta de mayor valor y en qué país se realizó?

🤖 RESPUESTA:
No tengo suficiente información en los datos para responder esto. El contexto solo muestra ventas de un mismo producto ("Ships") con el mismo valor ($0.05) en el mismo país (Australia). No hay datos de otras ventas con valores distintos para comparar.

📚 FUENTES UTILIZADAS: 3 documentos
   Fuente 1: Orden #0 - Australia - $0.05
   Fuente 2: Orden #0 - Australia - $0.05

📝 PRUEBA 2: ¿Qué países tienen ventas registradas y cuál es el monto promedio por país?

🤖 RESPUESTA:
**Respuesta basada en el contexto proporcionado:**

- **País con ventas registradas:** Australia
- **Monto promedio por país:** $0.38 (único registro disponible, correspondiente a Australia).

*Nota:* Solo hay datos de ventas para **Australia**, con un total de **$0.38** en la única venta registrada. No hay información suficiente para calcular promedios por otros países.

📚 FUENTES UTILIZADA

Chat Interactivo

In [59]:
def chat_ventas_interactivo():
    """
    Función interactiva para conversar con el sistema RAG
    """
    print("\n" + "="*60)
    print("💬 CHAT INTERACTIVO - SISTEMA RAG DE VENTAS")
    print("="*60)
    print("Bienvenido! Puedes hacer preguntas sobre ventas como:")
    print("  • '¿Cuáles son las ventas totales por país?'")
    print("  • 'Muéstrame los pedidos con mayor valor'")
    print("  • 'Qué patrones encuentras en los estados de pedidos?'")
    print("  • 'Dame un resumen de las ventas'")
    print("\nEscribe 'salir', 'exit' o 'quit' para terminar")
    print("Escribe 'estadisticas' para ver stats del sistema")
    print("="*60)

    while True:
        # Input del usuario
        pregunta = input("\n🔍 Tú: ").strip()

        # Comandos especiales
        if pregunta.lower() in ['salir', 'exit', 'quit']:
            print("\n👋 ¡Hasta luego! Gracias por usar el sistema RAG de ventas")
            break

        if pregunta.lower() == 'estadisticas':
            stats = rag_system.obtener_estadisticas()
            print(f"\n📊 Estadísticas del sistema:")
            print(f"  - Total consultas realizadas: {stats['total_consultas']}")
            print(f"  - Documentos indexados: {stats['documentos_indexados']}")
            print(f"  - Modelo utilizado: {stats['modelo']}")
            continue

        if not pregunta:
            continue

        # Procesar consulta RAG
        print("🤔 Procesando consulta...")
        resultado = rag_system.consultar(pregunta, mostrar_contexto=False)

        # Mostrar respuesta
        print(f"\n🤖 Asistente: {resultado['respuesta']}")

        # Opción para ver fuentes
        if len(resultado['fuentes']) > 0:
            ver_fuentes = input("\n📚 ¿Ver detalles de las fuentes usadas? (s/n): ").lower()
            if ver_fuentes == 's':
                print("\nFuentes consultadas:")
                for i, fuente in enumerate(resultado['fuentes'], 1):
                    print(f"  {i}. Orden #{fuente.get('order_number', 'N/A')}")
                    print(f"     País: {fuente.get('country', 'N/A')}")
                    print(f"     Venta: ${fuente.get('sales', 0):,.2f}")
                    print(f"     Estado: {fuente.get('status', 'N/A')}")

# Descomentar para iniciar chat
chat_ventas_interactivo()


💬 CHAT INTERACTIVO - SISTEMA RAG DE VENTAS
Bienvenido! Puedes hacer preguntas sobre ventas como:
  • '¿Cuáles son las ventas totales por país?'
  • 'Muéstrame los pedidos con mayor valor'
  • 'Qué patrones encuentras en los estados de pedidos?'
  • 'Dame un resumen de las ventas'

Escribe 'salir', 'exit' o 'quit' para terminar
Escribe 'estadisticas' para ver stats del sistema

🔍 Tú: ¿que pais tenes?
🤔 Procesando consulta...

🤖 Asistente: Australia.

📚 ¿Ver detalles de las fuentes usadas? (s/n): costos

🔍 Tú: salir

👋 ¡Hasta luego! Gracias por usar el sistema RAG de ventas


Validacion del Sistema

In [60]:
print("\n" + "="*60)
print("✅ VALIDACIÓN FINAL DEL SISTEMA RAG")
print("="*60)

# Verificar todos los componentes
validaciones = {
    "API Key": "✓" if API_KEY else "✗",
    "Cliente Mistral": "✓" if client else "✗",
    "Dataset cargado": "✓" if df is not None else "✗",
    "Documentos creados": f"{len(documentos)} documentos" if len(documentos) > 0 else "✗",
    "Vector store": f"{vectorstore._collection.count()} vectores" if vectorstore else "✗",
    "Sistema RAG": "✓" if rag_system else "✗"
}

print("\nComponentes del sistema:")
for componente, estado in validaciones.items():
    print(f"  {componente}: {estado}")

# Prueba final rápida
print("\n🔍 Realizando prueba final...")
prueba_final = rag_system.consultar("Dame un resumen general de las ventas")
print(f"Pregunta: DAME UN RESUMEN GENERAL DE LAS VENTAS")
print(f"Respuesta: {prueba_final['respuesta'][:200]}...")

print("\n" + "="*60)
print("🎉 SISTEMA RAG COMPLETAMENTE OPERATIVO")
print(f"📊 Total de documentos indexados: {vectorstore._collection.count()}")
print(f"🤖 Modelo: mistral-small-latest")
print(f"🔑 API Key: Configurada via Secret de Colab 'PruebaIA'")
print("="*60)


✅ VALIDACIÓN FINAL DEL SISTEMA RAG

Componentes del sistema:
  API Key: ✓
  Cliente Mistral: ✓
  Dataset cargado: ✓
  Documentos creados: 147 documentos
  Vector store: 588 vectores
  Sistema RAG: ✓

🔍 Realizando prueba final...
Pregunta: DAME UN RESUMEN GENERAL DE LAS VENTAS
Respuesta: **Resumen general de ventas:**

- **Total de ventas registradas:** 3 transacciones idénticas.
- **Producto vendido:** *Trucks and Buses* (Línea de producto: Trucks and Buses).
- **Cantidad promedio po...

🎉 SISTEMA RAG COMPLETAMENTE OPERATIVO
📊 Total de documentos indexados: 588
🤖 Modelo: mistral-small-latest
🔑 API Key: Configurada via Secret de Colab 'PruebaIA'
